In [1]:
# Identificaciones y par?metros personalizados confirmados por el grupo.
C1 = 1023092749
C2 = 1023092750
S = C1 + C2
nivel_confianza = {0: 0.90, 1: 0.95, 2: 0.99}[S % 3]
tau_ms = 180 + 10 * (S % 9)
epsilon = 0.01 * (1 + S % 3)
g = 1.4 + 0.1 * (S % 7)
t1_kb = 80 + 10 * (S % 5)
t2_kb = 400 + 25 * (S % 5)
pi = (4 + S % 6) / 1000
k_max = 4 + S % 5
r = S % 6
N = 10 + S % 11
print(f"S={S}, confianza={nivel_confianza:.0%}, tau={tau_ms} ms, epsilon={epsilon}, "
      f"g={g:.1f}, t1={t1_kb} KB, t2={t2_kb} KB, pi={pi}, k_max={k_max}, r={r}, N={N}")

S=2046185499, confianza=90%, tau=210 ms, epsilon=0.01, g=1.5, t1=120 KB, t2=500 KB, pi=0.007, k_max=8, r=3, N=16


# Partes A y B ? auditor?a, reg?menes y an?lisis

El r?gimen estable une [0,668) y [838,1200]. Consulte `documentos/calculso/actualizacion_A_B.md` para decisiones, f?rmulas y limitaciones.

In [2]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise FileNotFoundError("Ejecute el notebook desde la ra?z del proyecto o notebooks/")
sys.path.insert(0, str(ROOT / "src"))
from auditoria import audit
from regimenes import main as segmentar
from llegadas_intentos import main as analizar_b
from descriptivos import main as resumen_a3

## A1. Auditor?a
Se convierten los tama?os indicados y se excluyen filas inv?lidas siguiendo la tolerancia confirmada. Se conservan los motivos en las salidas.

In [3]:
solicitudes, conteos_auditoria = audit()

Filas originales: 34072; identificadores consolidados: 33971; válidas: 33737; excluidas: 234
duplicado exacto eliminado: 101
filas excluidas por datos de latencia inválidos o ausentes: 234
gpu_ms: ausente o contradictorio: 33
latencia_ms: inferior a red más GPU, fuera del redondeo: 67
latencia_ms: tiempo negativo: 101
red_ms: ausente o contradictorio: 34
tam_kb: convertido de bytes a KB: 50


## A2. Cambios de r?gimen
El an?lisis conserva BIC para uno a cinco segmentos y eval?a sensibilidad. El r?gimen elegido une los tramos [0,668) y [838,1200].

In [4]:
segmentar()

ancho=1s segmentos=[(0, 668, 16353), (668, 838, 8781), (838, 1200, 8603)] BIC=[-157630.6 -158319.5 -160866.9 -160854.2 -160843.2]


ancho=5s segmentos=[(0, 665, 16276), (665, 840, 8920), (840, 1200, 8541)] BIC=[-266227.5 -266914.2 -269408.5 -269398.7 -269389.9]
ancho=10s segmentos=[(0, 670, 16463), (670, 840, 8733), (840, 1200, 8541)] BIC=[-312997.6 -313669.8 -316172.7 -316165.4 -316157.6]
ancho=15s segmentos=[(0, 660, 16162), (660, 720, 2873), (720, 840, 6161), (840, 1200, 8541)] BIC=[-340356.3 -341029.4 -343446.2 -343447.  -343441.1]
ancho=30s segmentos=[(0, 660, 16162), (660, 720, 2873), (720, 840, 6161), (840, 1200, 8541)] BIC=[-387126.4 -387800.9 -390219.1 -390221.3 -390216.7]


Comparación de tramos: [{'prueba': 'KS tiempos entre llegadas', 'estadistico': np.float64(0.01420774466048258), 'p_valor': np.float64(0.20328224077138246)}, {'prueba': 'KS latencia', 'estadistico': np.float64(0.007467960713526456), 'p_valor': np.float64(0.9092863516753398)}, {'prueba': 'Mann-Whitney latencia', 'estadistico': np.float64(70673721.0), 'p_valor': np.float64(0.5402309621242771)}]


## A3. Estad?stica descriptiva
Por versi?n y r?gimen: varianza y desviaci?n muestrales con denominador n?1; asimetr?a y exceso de curtosis corregidos por sesgo.

In [5]:
resumen_a3()

Resumen A3: 16 combinaciones por variable, versión y régimen


## B. Llegadas, fallos y reintentos
Los intervalos entre llegadas se calculan por separado en los dos tramos estables. Se registran las salidas y los supuestos de las pruebas y simulaciones.

In [6]:
analizar_b(S)

{
  "B1": {
    "n": 24956,
    "duration_s": 1030,
    "rate_s": 24.229126213592235,
    "confidence": 0.9,
    "rate_interval": [
      23.977402453844743,
      24.482929810762933
    ],
    "mean_gap_s": 0.04127318666346076,
    "model_mean_gap_s": 0.04127263984612919,
    "model_mean_gap_interval_s": [
      0.040844784824747175,
      0.0417059354917593
    ],
    "numero_gaps": 24954,
    "max_gap_s": 0.48490000000015243,
    "ks_stat": 0.005051614741657384,
    "ks_bootstrap_p": 0.3516209476309227,
    "bootstrap_replicas": 400,
    "count_window_s": 10,
    "ventanas_completas": 102,
    "exposicion_excluida_s": 10,
    "fano": 0.8161327143077004,
    "dispersion_p": 0.17737514304197916,
    "ties": 42
  },
  "B2": [
    {
      "regimen": "estable",
      "solicitudes": 24956,
      "bloques": 249,
      "solicitudes_restantes": 56,
      "proporcion": 0.15298926109953517,
      "media_bloques": 15.301204819277109,
      "media_binomial": 15.298926109953518,
      "varianza_b

# Partes C y D ? avance
Se utilizan el r?gimen estable y los par?metros confirmados. Los apartados incompletos se identifican expl?citamente.

## C1. Distribuci?n mixta de G
FDA emp?rica y lognormal; verificaci?n de la varianza con divisor n y referencia muestral con n?1.

In [10]:
from tiempos_cd import c1
resultados_c1 = c1()

{
  "n": 24956,
  "n_cache": 10260,
  "n_positivos": 14696,
  "p_cache": 0.41112357749639367,
  "mu_positiva_ms": 53.288025993467606,
  "var_positiva_ddof0_ms2": 1365.8242988047136,
  "media_modelo_ms": 31.380062109312384,
  "media_empirica_ms": 31.380062109312387,
  "var_modelo_ddof0_ms2": 1491.7749942219798,
  "var_empirica_ddof0_ms2": 1491.77499422198,
  "var_muestral_ddof1_ms2": 1491.834772823231,
  "lognormal": {
    "mu_log_ms": 3.7648713104104856,
    "sigma_log": 0.6592925100386585,
    "log_verosimilitud": -70059.09260464262,
    "convergencia": true,
    "mensaje": "Optimization terminated successfully.",
    "gradiente_norma": 7.328604253514419e-16
  }
}


## C2. Ajuste de G positivo
Optimizaci?n BFGS de log-verosimilitudes Gamma y lognormal; comparaci?n Gamma por momentos y distancias KS descriptivas. El ajuste marginal no implica independencia. Un cambio en la mezcla de tama?os puede modificar la cola; la direcci?n requiere evidencia adicional.

In [7]:
from tiempos_c2 import main as ajustar_c2
resultados_c2 = ajustar_c2()

{
  "n_sin_cache": 14696,
  "gamma": {
    "k": 2.5256598603388554,
    "theta_ms": 21.098655032840576,
    "log_verosimilitud": -70333.5700044394,
    "convergencia": true,
    "mensaje": "Optimization terminated successfully.",
    "gradiente_norma": 6.52616509373158e-09,
    "momentos_k": 2.078906117856119,
    "momentos_theta_ms": 25.632723640460064,
    "ks_distancia": 0.03975614700695762
  },
  "lognormal": {
    "mu_log_ms": 3.7648713104104856,
    "sigma_log": 0.6592925100386585,
    "log_verosimilitud": -70059.09260464262,
    "convergencia": true,
    "mensaje": "Optimization terminated successfully.",
    "gradiente_norma": 7.328604253514419e-16,
    "ks_distancia": 0.01071893262702378
  },
  "nota_ks": "Distancia descriptiva con parámetros estimados en los mismos datos; no es una prueba calibrada."
}


## C3. Tiempo de red
El valor p de KS es nominal. Para Anderson?Darling se reporta la referencia 0,656 elegida por el estudiante y, por separado, el valor de SciPy instalado.

In [11]:
from tiempos_cd import c3
resultados_c3 = c3(seed=S)

F:\Universidad\CienciaComputacionaIntermedia\Proyecto Final V2\CCI-ProyectoFinal\src\tiempos_cd.py:84: FutureWarning: As of SciPy 1.17, users must choose a p-value calculation method by providing the `method` parameter. `method='interpolate'` interpolates the p-value from pre-calculated tables; `method` may also be an instance of `MonteCarloMethod` to approximate the p-value via Monte Carlo simulation. When `method` is specified, the result object will include a `pvalue` attribute and not attributes `critical_value`, `significance_level`, or `fit_result`. Beginning in 1.19.0, these other attributes will no longer be available, and a p-value will always be computed according to one of the available `method` options.
  ad = stats.anderson(red, dist="norm")


{
  "n": 24956,
  "scipy_version": "1.18.1",
  "media_ms": 29.610606267029976,
  "desviacion_muestral_ms": 4.6058674649054865,
  "asimetria_corregida": -0.0038977867846973838,
  "exceso_curtosis_corregido": -0.019293692261624206,
  "KS_D": 0.004552755850951273,
  "KS_p_nominal_parametros_estimados": 0.6772772237947604,
  "AD_estadistico": 0.5910893573018257,
  "AD_valores_criticos": [
    0.561,
    0.631,
    0.752,
    0.873,
    1.035
  ],
  "AD_significancias_pct": [
    15.0,
    10.0,
    5.0,
    2.5,
    1.0
  ],
  "colas": [
    {
      "umbral_ms": 35,
      "empirica": 0.12205481647699952,
      "normal": 0.12097738467251001
    },
    {
      "umbral_ms": 40,
      "empirica": 0.012101298284981568,
      "normal": 0.012045131245645787
    },
    {
      "umbral_ms": 45,
      "empirica": 0.0004007052412245552,
      "normal": 0.0004169977020784897
    }
  ],
  "AD_critico_10_scipy_instalado": 0.631,
  "AD_referencia_10_indicada_por_estudiante": 0.656,
  "AD_decision_referen

## C4. Tiempo total de GPU
Dos aproximaciones y 200.000 r?plicas con remuestreo i.i.d. de la distribuci?n emp?rica.

In [12]:
from tiempos_cd import c4
resultados_c4 = c4(seed=S)

{
  "N": 16,
  "n_positivos": 14696,
  "media_ms": 53.288025993467606,
  "var_muestral_ms2": 1365.9172436362076,
  "umbral_ms": 1092.6084158954818,
  "prob_normal": 0.05224656629413653,
  "prob_gamma": 0.04494678957869444,
  "prob_remuestreo_iid": 0.063425,
  "replicas": 200000,
  "IC_monte_carlo_90": [
    0.06253075333640754,
    0.06432860926314318
  ],
  "nota": "El remuestreo i.i.d. usa la distribución empírica; no preserva dependencia temporal ni prueba que sea la distribución poblacional."
}


## D1. Distribuci?n conjunta e independencia
Tablas, contraste y los intervalos Wald al 90 % indicados por el estudiante.

In [13]:
from dependencias_d1 import main as analizar_d1
resultados_d1 = analizar_d1()

{
  "n": 24956,
  "marginal_clase": {
    "pequeña": 0.2656275044077576,
    "mediana": 0.6543516589196987,
    "grande": 0.08002083667254367
  },
  "marginal_cache": {
    "0": 0.5888764225036064,
    "1": 0.4111235774963936
  },
  "chi_cuadrado": 950.1616336695627,
  "grados_libertad": 2,
  "p_valor": 4.731760892733954e-207,
  "frecuencia_esperada_minima": 821.0137842602982,
  "intervalos": "Wald normal al 90 %; solicitado por el estudiante."
}


## D2. Esperanza y varianza condicional
Identidades con divisor n, 25 grupos por cuantiles y LOWESS lineal local tric?bico con frac=0,15, sin iteraciones robustas.

In [14]:
from esperanza_d2 import main as analizar_d2
resultados_d2 = analizar_d2()

{
  "todas": {
    "n": 24956,
    "media_ms": 31.380062109312387,
    "var_empirica_ddof0_ms2": 1491.77499422198,
    "var_muestral_ddof1_ms2": 1491.834772823231,
    "esperanza_var_condicional_ms2": 1171.7031935309665,
    "var_esperanza_condicional_ms2": 320.0718006910166,
    "suma_componentes_ms2": 1491.774994221983,
    "fraccion_explicada": 0.214557692635106,
    "comprobacion_media_ms": 31.380062109312387
  },
  "sin_cache": {
    "n": 14696,
    "media_ms": 53.288025993467606,
    "var_empirica_ddof0_ms2": 1365.8242988047136,
    "var_muestral_ddof1_ms2": 1365.9172436362076,
    "esperanza_var_condicional_ms2": 903.6840221136522,
    "var_esperanza_condicional_ms2": 462.1402766910629,
    "suma_componentes_ms2": 1365.824298804715,
    "fraccion_explicada": 0.33835997580032806,
    "comprobacion_media_ms": 53.288025993467606
  },
  "metodo_continuo": "25 grupos por cuantiles y LOWESS lineal local tricúbico",
  "cuantiles_solicitados": 25,
  "cuantiles_obtenidos": 25,
  "lowess_

## D3. Regresi?n matricial y residuos
OLS y WLS con errores cl?sicos y HC3; intervalos del 90 % con t de Student y n?2 grados de libertad.

In [15]:
from regresion_d3 import main as analizar_d3
resultados_d3 = analizar_d3()

{
  "n": 14696,
  "ols": {
    "beta": [
      15.099850275109008,
      0.14526892246466697
    ],
    "se_clasico": [
      0.39384602380718475,
      0.001243814582040756
    ],
    "se_hc3": [
      0.43868081228458217,
      0.002040253520915489
    ],
    "max_leverage": 0.00506864567088802,
    "condicion_matriz": 322643.64628190297
  },
  "wls": {
    "beta": [
      14.948770574566511,
      0.1447036754760385
    ],
    "se_clasico": [
      0.08295735976614729,
      0.0012591930463721592
    ],
    "se_hc3": [
      1.010697508625926,
      0.0068700937661319616
    ],
    "max_leverage": 0.31431734610212064,
    "condicion_matriz": 5573.863662580394
  },
  "breusch_pagan_lm": 1874.881655477428,
  "bp_gl": 1,
  "bp_p_numerico": 0.0,
  "intervalos": {
    "nivel": 0.9,
    "distribucion": "t de Student",
    "grados_libertad": 14694,
    "valor_critico": 1.6449573336211347
  },
  "nota": "p numérico cero puede indicar subdesbordamiento; WLS asume la forma de pesos elegida."


## D4. Covarianza y correlaci?n
Se calcula la covarianza muestral y la correlaci?n de Pearson de las solicitudes estables no cacheadas.

In [9]:
from dependencias_d4 import main as analizar_d4
resultados_d4 = analizar_d4()

{
  "n": 14696,
  "covarianza_muestral_kb_ms": 4526.56130844586,
  "pearson_r": 0.6938384563730369,
  "pearson_p_numerico": 0.0,
  "nota_p": "Si p se imprime como 0, se trata de subdesbordamiento numérico, no probabilidad exactamente nula.",
  "r_cuadrado": 0.4814118035421186
}


# Partes E, F y G ? resultados provisionales
Las discrepancias y decisiones pendientes est?n en `documentos/calculos/discrepancias_pendientes.md`.

## E. Monitoreo
Wilson, Bayes, independencia condicional y falsas alarmas. Las discrepancias del material aportado y la frecuencia esperada baja en E3 requieren revisi?n.

In [16]:
import pandas as pd
from parte_e import analyze as analizar_e
resultados_e = analizar_e(pd.read_csv(ROOT / "data/raw/monitoreo.csv"), seed=S, output_root=ROOT / "outputs")

## F. Inferencia y validaci?n
Se guardan ambas convenciones de cobertura en F1. El bootstrap por bloques de F2, la rejilla F4 y los c?lculos F5 se documentan con sus diferencias frente a las cifras aportadas.

In [17]:
from parte_f import main as analizar_f
analizar_f(seed=S)

Filas originales: 29781; identificadores consolidados: 29692; válidas: 29485; excluidas: 207
duplicado exacto eliminado: 88
filas excluidas por datos de latencia inválidos o ausentes: 207
gpu_ms: ausente o contradictorio: 29
identificador con registros diferentes: 1
latencia_ms: ausente o contradictorio: 1
latencia_ms: inferior a red más GPU, fuera del redondeo: 59
latencia_ms: tiempo negativo: 88
red_ms: ausente o contradictorio: 30
tam_kb: convertido de bytes a KB: 44


## G1?G3. Simulaci?n FIFO y contraste parcial
Los resultados est?n condicionados al modelo. G1 muestra una discrepancia en la cola; la propagaci?n de incertidumbre de G3 sigue pendiente de m?todo.

In [18]:
from parte_g import main as simular_g
resultados_g = simular_g(seed=S)

                metrica  observado   simulado  ic_90_inferior  ic_90_superior  diferencia_simulado_menos_observado  replicas
             mediana_ms   53.20500  53.223692       53.046030       53.401354                             0.018692        20
                 p95_ms  152.20250 152.578265      152.084474      153.072056                             0.375765        20
proporcion_mayor_210_ms    0.01138   0.013588        0.013331        0.013844                             0.002208        20
 gpus      rho  violacion  ic_90_inferior  ic_90_superior  replicas  duracion_s  calentamiento_s  estable_rho_menor_1
    1 1.147896   0.588579        0.587568        0.589591        30         600               30                False
    2 0.573948   0.030694        0.029804        0.031585        30         600               30                 True
    3 0.382632   0.010189        0.009943        0.010435        30         600               30                 True
    4 0.286974   0.009057   

## G4. Memorando provisional
El documento se genera como borrador porque siguen pendientes decisiones y validaciones.

In [19]:
from memorando_g4 import main as generar_memorando
memoria_g4 = generar_memorando()

Memorando G4 provisional generado en Markdown y PDF de una página
